In [1]:
import os

# Kaggle input directory list
print(os.listdir("/kaggle/input"))


['brats-2021-task1']


In [4]:
import tarfile
import os

INPUT_PATH = "/kaggle/input/brats-2021-task1"
OUTPUT_PATH = "/kaggle/working/brats2021"

os.makedirs(OUTPUT_PATH, exist_ok=True)

tar_path = os.path.join(INPUT_PATH, "BraTS2021_Training_Data.tar")

with tarfile.open(tar_path) as tar:
    tar.extractall(path=OUTPUT_PATH)

print("Extraction done!")


Extraction done!


In [5]:
print(os.listdir(OUTPUT_PATH))


['BraTS2021_00642', 'BraTS2021_00261', 'BraTS2021_01368', 'BraTS2021_00405', 'BraTS2021_00123', 'BraTS2021_01062', 'BraTS2021_00108', 'BraTS2021_00694', 'BraTS2021_01122', 'BraTS2021_00219', 'BraTS2021_00243', 'BraTS2021_00311', 'BraTS2021_00480', 'BraTS2021_00045', 'BraTS2021_01077', 'BraTS2021_00646', 'BraTS2021_01109', 'BraTS2021_01070', 'BraTS2021_00750', 'BraTS2021_00765', 'BraTS2021_00386', 'BraTS2021_00090', 'BraTS2021_00380', 'BraTS2021_01584', 'BraTS2021_00166', 'BraTS2021_01611', 'BraTS2021_01424', 'BraTS2021_00452', 'BraTS2021_01140', 'BraTS2021_01535', 'BraTS2021_01427', 'BraTS2021_01363', 'BraTS2021_00668', 'BraTS2021_01553', 'BraTS2021_01587', 'BraTS2021_00059', 'BraTS2021_01165', 'BraTS2021_01161', 'BraTS2021_01210', 'BraTS2021_00810', 'BraTS2021_00068', 'BraTS2021_01207', 'BraTS2021_00061', 'BraTS2021_00426', 'BraTS2021_00698', 'BraTS2021_00382', 'BraTS2021_00768', 'BraTS2021_00414', 'BraTS2021_01119', 'BraTS2021_01606', 'BraTS2021_00787', 'BraTS2021_01306', 'BraTS2021_

In [7]:
import os

print(os.path.exists("/kaggle/working/brats2021"))
print(os.listdir("/kaggle/working"))


True
['.virtual_documents', 'brats2021']


In [9]:
TRAIN_ROOT = "/kaggle/working/brats2021/BraTS2021_Training_Data"


In [12]:
TRAIN_ROOT = "/kaggle/working/brats2021"


In [13]:
import os

TRAIN_ROOT = "/kaggle/working/brats2021"

patients = sorted([
    p for p in os.listdir(TRAIN_ROOT)
    if p.startswith("BraTS2021_")
])

print("Total patients:", len(patients))
print("Sample patients:", patients[:5])


Total patients: 1251
Sample patients: ['BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003', 'BraTS2021_00005', 'BraTS2021_00006']


In [14]:
p = patients[0]
p_path = os.path.join(TRAIN_ROOT, p)

print(p_path)
print(os.listdir(p_path))


/kaggle/working/brats2021/BraTS2021_00000
['BraTS2021_00000_t1ce.nii.gz', 'BraTS2021_00000_seg.nii.gz', 'BraTS2021_00000_t1.nii.gz', 'BraTS2021_00000_flair.nii.gz', 'BraTS2021_00000_t2.nii.gz']


In [16]:
"""
Advanced Ensemble Framework for BraTS 2021 Brain Tumor Classification
Combining EfficientNet (CNN) + Vision Transformer (ViT) with Novel Fusion

Dataset: /kaggle/working/brats2021/
Structure: BraTS2021_XXXXX/ with t1ce, t1, t2, flair, seg.nii.gz files
"""

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import nibabel as nib
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# ============================================================================
# REPRODUCIBILITY
# ============================================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    # Paths
    train_root = "/kaggle/working/brats2021"
    
    # Training
    img_size = 224
    batch_size = 8
    epochs = 30
    lr = 1e-4
    weight_decay = 1e-4
    
    # Model
    num_classes = 4
    cnn_feature_dim = 1280
    vit_feature_dim = 768
    fusion_dim = 512
    mc_dropout_samples = 10
    dropout_rate = 0.3

config = Config()

# ============================================================================
# BRATS 2021 DATASET
# ============================================================================
class BraTSDataset(Dataset):
    def __init__(self, patient_folders, labels, transform=None):
        self.patient_folders = patient_folders
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.patient_folders)
    
    def load_and_normalize(self, filepath, slice_idx):
        """Load NIfTI file and extract normalized slice"""
        try:
            img = nib.load(filepath).get_fdata()
            # Get middle slice
            if slice_idx >= img.shape[2]:
                slice_idx = img.shape[2] // 2
            slice_data = img[:, :, slice_idx]
            # Normalize to [0, 1]
            if slice_data.max() > slice_data.min():
                slice_data = (slice_data - slice_data.min()) / (slice_data.max() - slice_data.min())
            return slice_data
        except Exception as e:
            print(f"Error loading {filepath}: {e}")
            return np.zeros((240, 240))
    
    def __getitem__(self, idx):
        patient_folder = self.patient_folders[idx]
        label = self.labels[idx]
        
        try:
            # Get patient ID from folder name
            patient_id = os.path.basename(patient_folder)
            
            # Construct file paths
            t1ce_path = os.path.join(patient_folder, f"{patient_id}_t1ce.nii.gz")
            t2_path = os.path.join(patient_folder, f"{patient_id}_t2.nii.gz")
            flair_path = os.path.join(patient_folder, f"{patient_id}_flair.nii.gz")
            
            # Load first modality to get shape
            img = nib.load(t1ce_path).get_fdata()
            slice_idx = img.shape[2] // 2  # Middle slice
            
            # Load all three modalities
            t1ce_slice = self.load_and_normalize(t1ce_path, slice_idx)
            t2_slice = self.load_and_normalize(t2_path, slice_idx)
            flair_slice = self.load_and_normalize(flair_path, slice_idx)
            
            # Stack as RGB channels
            image = np.stack([t1ce_slice, t2_slice, flair_slice], axis=-1)
            
            # Convert to uint8 for PIL
            from PIL import Image
            image = (image * 255).astype(np.uint8)
            image = Image.fromarray(image)
            
            if self.transform:
                image = self.transform(image)
            
            return image, label
            
        except Exception as e:
            print(f"Error in __getitem__ for {patient_folder}: {e}")
            # Return dummy tensor
            from PIL import Image
            dummy = Image.fromarray(np.zeros((240, 240, 3), dtype=np.uint8))
            if self.transform:
                dummy = self.transform(dummy)
            return dummy, label

# ============================================================================
# DATA PREPARATION
# ============================================================================
def prepare_brats_data(train_root):
    """Load all patient folders and create labels"""
    print("="*70)
    print("LOADING BRATS 2021 DATASET")
    print("="*70)
    
    # Get all patient folders
    patients = sorted([
        p for p in os.listdir(train_root)
        if p.startswith("BraTS2021_")
    ])
    
    print(f"Total patients found: {len(patients)}")
    print(f"Sample patients: {patients[:5]}")
    
    patient_paths = []
    labels = []
    
    for patient in tqdm(patients, desc="Processing patients"):
        patient_path = os.path.join(train_root, patient)
        
        # Check if segmentation exists
        seg_file = os.path.join(patient_path, f"{patient}_seg.nii.gz")
        
        if os.path.exists(seg_file):
            try:
                # Load segmentation
                seg = nib.load(seg_file).get_fdata()
                unique_vals = np.unique(seg)
                
                # Create label based on tumor composition
                # BraTS labels: 0=background, 1=NCR, 2=ED, 4=ET
                if 4 in unique_vals:  # Enhancing tumor present
                    label = 3  # High-grade/aggressive
                elif 2 in unique_vals and 1 in unique_vals:  # Edema + Necrosis
                    label = 2  # Medium-grade
                elif 2 in unique_vals:  # Only edema
                    label = 1  # Low-grade
                else:
                    label = 0  # Minimal/no tumor
                
                patient_paths.append(patient_path)
                labels.append(label)
                
            except Exception as e:
                print(f"Error processing {patient}: {e}")
                continue
        else:
            print(f"No segmentation for {patient}, skipping...")
    
    print(f"\n✓ Valid patients: {len(patient_paths)}")
    print(f"Label distribution:")
    for i, count in enumerate(np.bincount(labels)):
        print(f"  Class {i}: {count} samples")
    
    return patient_paths, labels

# ============================================================================
# MODEL COMPONENTS
# ============================================================================
class CrossModalAttention(nn.Module):
    def __init__(self, cnn_dim, vit_dim, hidden_dim=256, num_heads=8):
        super().__init__()
        self.cnn_proj = nn.Linear(cnn_dim, hidden_dim)
        self.vit_proj = nn.Linear(vit_dim, hidden_dim)
        self.cross_attn_cnn_to_vit = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=0.1, batch_first=True
        )
        self.cross_attn_vit_to_cnn = nn.MultiheadAttention(
            hidden_dim, num_heads, dropout=0.1, batch_first=True
        )
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        
    def forward(self, cnn_feat, vit_feat):
        cnn_proj = self.cnn_proj(cnn_feat).unsqueeze(1)
        vit_proj = self.vit_proj(vit_feat).unsqueeze(1)
        cnn_enhanced, _ = self.cross_attn_cnn_to_vit(cnn_proj, vit_proj, vit_proj)
        cnn_enhanced = self.ln1(cnn_enhanced + cnn_proj).squeeze(1)
        vit_enhanced, _ = self.cross_attn_vit_to_cnn(vit_proj, cnn_proj, cnn_proj)
        vit_enhanced = self.ln2(vit_enhanced + vit_proj).squeeze(1)
        return cnn_enhanced, vit_enhanced

class AdaptiveFusionGate(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.gate_network = nn.Sequential(
            nn.Linear(feature_dim * 2, feature_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(feature_dim, 2),
            nn.Softmax(dim=1)
        )
        
    def forward(self, cnn_feat, vit_feat):
        combined = torch.cat([cnn_feat, vit_feat], dim=1)
        weights = self.gate_network(combined)
        cnn_weight = weights[:, 0:1]
        vit_weight = weights[:, 1:2]
        fused = cnn_weight * cnn_feat + vit_weight * vit_feat
        return fused, weights

class UncertaintyHead(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_rate=0.3):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.fc1 = nn.Linear(input_dim, 256)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.fc3 = nn.Linear(128, num_classes)
        
    def forward(self, x, mc_dropout=False):
        x = F.relu(self.fc1(x))
        x = self.dropout1(x) if (self.training or mc_dropout) else x
        x = F.relu(self.fc2(x))
        x = self.dropout2(x) if (self.training or mc_dropout) else x
        x = self.fc3(x)
        return x
    
    def predict_with_uncertainty(self, x, n_samples=10):
        self.eval()
        predictions = []
        for _ in range(n_samples):
            with torch.no_grad():
                pred = self.forward(x, mc_dropout=True)
                predictions.append(F.softmax(pred, dim=1))
        predictions = torch.stack(predictions)
        mean_pred = predictions.mean(dim=0)
        uncertainty = predictions.var(dim=0).mean(dim=1)
        return mean_pred, uncertainty

class HybridEnsemble(nn.Module):
    def __init__(self, num_classes=4, pretrained=True):
        super().__init__()
        self.efficientnet = timm.create_model(
            'efficientnet_b0', pretrained=pretrained, 
            num_classes=0, global_pool=''
        )
        self.vit = timm.create_model(
            'vit_base_patch16_224', pretrained=pretrained, num_classes=0
        )
        self.cnn_pool = nn.AdaptiveAvgPool2d(1)
        self.cross_attention = CrossModalAttention(
            cnn_dim=config.cnn_feature_dim,
            vit_dim=config.vit_feature_dim,
            hidden_dim=config.fusion_dim
        )
        self.fusion_gate = AdaptiveFusionGate(config.fusion_dim)
        self.classifier = UncertaintyHead(
            config.fusion_dim, num_classes, dropout_rate=config.dropout_rate
        )
        
    def forward(self, x):
        cnn_feat = self.efficientnet(x)
        cnn_feat = self.cnn_pool(cnn_feat).flatten(1)
        vit_feat = self.vit.forward_features(x)
        vit_feat = vit_feat[:, 0]
        cnn_enhanced, vit_enhanced = self.cross_attention(cnn_feat, vit_feat)
        fused_feat, _ = self.fusion_gate(cnn_enhanced, vit_enhanced)
        logits = self.classifier(fused_feat)
        return logits
    
    def predict_with_uncertainty(self, x):
        self.eval()
        with torch.no_grad():
            cnn_feat = self.efficientnet(x)
            cnn_feat = self.cnn_pool(cnn_feat).flatten(1)
            vit_feat = self.vit.forward_features(x)
            vit_feat = vit_feat[:, 0]
            cnn_enhanced, vit_enhanced = self.cross_attention(cnn_feat, vit_feat)
            fused_feat, _ = self.fusion_gate(cnn_enhanced, vit_enhanced)
        mean_pred, uncertainty = self.classifier.predict_with_uncertainty(
            fused_feat, n_samples=config.mc_dropout_samples
        )
        return mean_pred, uncertainty

class FocalLossWithSmoothing(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, smoothing=0.1):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
    
    def forward(self, logits, target):
        log_probs = F.log_softmax(logits, dim=-1)
        probs = torch.exp(log_probs)
        n_classes = logits.size(-1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        focal_weight = (1.0 - probs) ** self.gamma
        loss = -self.alpha * focal_weight * true_dist * log_probs
        return loss.sum(dim=1).mean()

# ============================================================================
# DATALOADERS
# ============================================================================
def get_dataloaders(patient_paths, labels, config):
    train_transform = transforms.Compose([
        transforms.Resize((config.img_size, config.img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    eval_transform = transforms.Compose([
        transforms.Resize((config.img_size, config.img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Split data
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        patient_paths, labels, test_size=0.3, random_state=SEED, stratify=labels
    )
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels, test_size=0.5, random_state=SEED, stratify=temp_labels
    )
    
    train_dataset = BraTSDataset(train_paths, train_labels, transform=train_transform)
    val_dataset = BraTSDataset(val_paths, val_labels, transform=eval_transform)
    test_dataset = BraTSDataset(test_paths, test_labels, transform=eval_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size,
                             shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size,
                           shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size,
                            shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, test_loader

# ============================================================================
# TRAINING
# ============================================================================
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total:.4f}'})
    
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return running_loss / total, correct / total

def comprehensive_evaluation(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_uncertainties = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating"):
            images = images.to(device)
            probs, uncertainty = model.predict_with_uncertainty(images)
            preds = torch.argmax(probs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_uncertainties.extend(uncertainty.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_uncertainties = np.array(all_uncertainties)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    print("\n" + "="*70)
    print("TEST SET EVALUATION")
    print("="*70)
    print(f"Accuracy:  {accuracy*100:.2f}%")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"Mean Uncertainty: {all_uncertainties.mean():.4f}")
    
    class_names = ['Minimal', 'Low-Grade', 'Medium-Grade', 'High-Grade']
    print("\nPer-Class Report:")
    print(classification_report(all_labels, all_preds, 
                                target_names=class_names, digits=4, zero_division=0))
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix - BraTS 2021')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('brats_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return {'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1}

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("\n" + "="*70)
    print("BRATS 2021 BRAIN TUMOR CLASSIFICATION")
    print("Hybrid Ensemble: EfficientNet + Vision Transformer")
    print("="*70)
    
    # Load data
    patient_paths, labels = prepare_brats_data(config.train_root)
    
    if len(patient_paths) == 0:
        print("❌ No valid data found!")
        return None, None
    
    # Create dataloaders
    print("\n" + "="*70)
    print("CREATING DATALOADERS")
    print("="*70)
    train_loader, val_loader, test_loader = get_dataloaders(patient_paths, labels, config)
    print(f"Train samples: {len(train_loader.dataset)}")
    print(f"Val samples: {len(val_loader.dataset)}")
    print(f"Test samples: {len(test_loader.dataset)}")
    
    # Build model
    print("\n" + "="*70)
    print("BUILDING MODEL")
    print("="*70)
    model = HybridEnsemble(num_classes=config.num_classes).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")
    
    # Training
    criterion = FocalLossWithSmoothing(alpha=1.0, gamma=2.0, smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs)
    
    print("\n" + "="*70)
    print("TRAINING")
    print("="*70)
    best_val_acc = 0.0
    
    for epoch in range(1, config.epochs + 1):
        print(f"\nEpoch {epoch}/{config.epochs}")
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()
        
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc*100:.2f}%")
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc*100:.2f}%")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_brats_model.pth')
            print(f"✓ Model saved! Best Val Acc: {best_val_acc*100:.2f}%")
    
    # Final evaluation
    model.load_state_dict(torch.load('best_brats_model.pth'))
    results = comprehensive_evaluation(model, test_loader, device)
    
    print("\n" + "="*70)
    print("✅ TRAINING COMPLETE!")
    print("="*70)
    print(f"Best Val Acc: {best_val_acc*100:.2f}%")
    print(f"Test Acc: {results['accuracy']*100:.2f}%")
    print("="*70)
    
    return model, results

if __name__ == "__main__":
    model, results = main()

Using device: cuda

BRATS 2021 BRAIN TUMOR CLASSIFICATION
Hybrid Ensemble: EfficientNet + Vision Transformer
LOADING BRATS 2021 DATASET
Total patients found: 1251
Sample patients: ['BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003', 'BraTS2021_00005', 'BraTS2021_00006']


Processing patients: 100%|██████████| 1251/1251 [14:21<00:00,  1.45it/s]



✓ Valid patients: 1251
Label distribution:
  Class 0: 0 samples
  Class 1: 6 samples
  Class 2: 27 samples
  Class 3: 1218 samples

CREATING DATALOADERS
Train samples: 875
Val samples: 188
Test samples: 188

BUILDING MODEL


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

...

Training: 100%|██████████| 110/110 [02:45<00:00,  1.50s/it, loss=0.2166, acc=0.9920]


Train - Loss: 0.2281, Acc: 99.20%
Val   - Loss: 0.2416, Acc: 97.87%


Evaluating: 100%|██████████| 24/24 [00:37<00:00,  1.55s/it]


TEST SET EVALUATION
Accuracy:  97.34%
Precision: 0.9631
Recall:    0.9734
F1-Score:  0.9673
Mean Uncertainty: 0.0000

Per-Class Report:


ValueError: Number of classes, 3, does not match size of target_names, 4. Try specifying the labels parameter